# Ayudantía N°3:
## Mutate 2, summarise y group by + data wrangling



## Antes de comenzar: trabajar con archivos en Google Colab

Esta ayudantía está preparada para ejecutarse en **Google Colab utilizando el entorno de ejecución de R**

Primero, sube a Colab el archivo:

`Encuesta-Estudiantes-Antropología-2023-(respuestas).xlsx`

Puedes hacerlo ejecutando la siguiente celda y seleccionando el archivo que descargaste en la página del curso desde tu computador

In [ ]:
# Subir el archivo Excel a Google Colab
# Ejecuta esta celda y selecciona:
# Encuesta-Estudiantes-Antropología-2023-(respuestas).xlsx

library(googledrive) # solo si trabajas con Google Drive

# Alternativa sencilla: usar el botón "Archivos" de Colab
# y subir manualmente el archivo a la carpeta de trabajo.

### Nota para Colab

Si subes el archivo mediante el panel **Archivos** de Colab, este quedará disponible en la carpeta de trabajo y podrás utilizar directamente el nombre del archivo en `read.xlsx()`.

Si Colab te solicita instalar algún paquete, ejecuta la celda de instalación correspondiente.

## Instalar y cargar paquetes necesarios

In [ ]:
# Instalar paquetes necesarios
install.packages(c(
  "tidyverse",
  "openxlsx",
  "readxl",
  "janitor",
  "writexl",
  "DataExplorer",
  "stringi"
))

# Cargar paquetes
library(tidyverse)
library(openxlsx)
library(readxl)
library(janitor)      # limpieza de datos
library(writexl)     # guardar tablas formato Excel
library(DataExplorer) # exploración rápida
library(stringi)

In [ ]:
# Con pacman es más fácil
install.packages("pacman")
pacman::p_load(tidyverse, readxl, openxlsx, janitor, writexl, DataExplorer, stringi)

## Importación de la base de datos

In [ ]:
base_antropologia <- read.xlsx("Encuesta-Estudiantes-Antropología-2023-(respuestas).xlsx") %>%
  dplyr::select(3:ncol(.)) # seleccionar desde la columna 3 en adelante para omitir información personal de encuestados (confidencialidad)

# Cómo se llama nuestra bdbb?

# I. **Mutate, summarise y group by**
Cuando trabajamos con una base de datos, no solamente podemos cambiar nombres o seleccionar variables. También podemos **crear nuevas variables, resumir información y realizar esos resúmenes para distintos grupos**



# 1. `mutate()`: crear y transformar variables

Sabemos que `mutate()` nos permite crear una nueva variable a partir de información que ya tenemos en nuestra base de datos, o modificar una variable existente

- ### Estructura del código:

    `base <- base %>% mutate(variable_nueva = case_when(variable_antigua sea X categoría_antigua ~ categoría_nueva)`

Por ejemplo, nuestra variable *edad* contiene respuestas como:

*19*

*22*

*20*

*21*

*30 años*

*23*

*22 años*

Como podemos observar, la variable no está completamente estandarizada: algunas respuestas contienen solamente el número y otras contienen la palabra *"años"*

#### Podemos utilizar `mutate()` para crear una nueva variable de edad numérica

In [ ]:
base_antropologia <- base_antropologia %>%

  mutate(edad_num = as.numeric(gsub(" años", "", edad))

  )

# edad_num: nombre de la nueva variable
# as.numeric(): convierte el resultado en números
# gsub(" años", "", edad): elimina la palabra " años"

De esta manera, podemos trabajar posteriormente con la edad como una variable numérica

In [ ]:
# Veamos cómo queda aplicando la función table()
table(base_antropologia$edad_num)

## 2. `summarise()`: resumir información

Mientras que mutate() sirve para crear o transformar variables, summarise() sirve para obtener un **resumen de los datos**

- ### Estructura del código:

    ```
    base_de_datos %>%

  summarise(

    nombre_resumen = función_calculo(variable)

  )

In [ ]:
# Ejemplo para calcular media/promedio con variable edad numérica creada anteriormente

base_antropologia %>%

  summarise(

    edad_promedio = mean(edad_num, na.rm = TRUE)

  )

## 3. `group_by()`: analizar por grupos

Aquí aparece una de las herramientas más importantes para el análisis de datos

group_by() permite **agrupar temporalmente los datos según una variable**

- ### Estructura del código:

    ```
    base_de_datos %>%

  group_by(variable)

In [ ]:
# Ejemplo agrupar según género los datos de la base

  base_antropologia %>%

  group_by(genero) %>%

  summarise(edad_promedio = mean(edad_num, na.rm = TRUE) #calcular promedio dentro de cada grupo

  )

   #nos entrega un resumen de esta agrupación de datos

In [ ]:
# Agrupamos las personas según género y calculamos la edad promedio

base_antropologia %>%

  group_by(genero) %>%

  summarise(

    edad_promedio = mean(
      edad_num,
      na.rm = TRUE
    )

  )

## 4. `mutate()` + `case_when()`

mutate() se vuelve especialmente útil cuando queremos **crear categorías a partir de una variable existente**

Por ejemplo, podemos transformar la **edad en grupos**:

In [ ]:
base_antropologia <- base_antropologia %>%

  mutate(

    grupo_edad = case_when(

      edad_num < 21 ~ "19-20 años",

      edad_num >= 21 & edad_num <= 23 ~ "21-23 años",

      edad_num >= 24 ~ "24 años o más"

    )

  )

### Estructura de `case_when`:

```
case_when(

  condición ~ resultado,

  condición ~ resultado,

  condición ~ resultado

)

# II. **Data wrangling**
**Data wrangling** es el proceso de limpiar, transformar y organizar datos crudos para que estén listos para su análisis.

En el proceso de data wrangling realizaremos:

1. Exploración de la base de datos: `glimpse` y `names`
2. Limpieza de datos: limpiar y renombrar variables `janitor`, `mutate` + `rename`
3. Transformación de datos: recodificación de categorías `recode` y `case_when`


## 1. Explorar la base de datos
### Vistazo inicial

In [ ]:
glimpse(base_antropologia)

### Ver nombres de las variables

In [ ]:
names(base_antropologia)

## 2. Limpieza de datos: Renombrar variables
- `janitor::clean_names()`
- `rename()`


### a) Limpiar nombres de variables

In [ ]:
base_antropologia <- janitor::clean_names(base_antropologia)
#estandariza los nombres (mayúsculas a minúsculas, reemplaza espacios y elimina o transforma caracteres problemáticos)

names(base_antropologia)

##### Se pueden **copiar y pegar** los nombres resultantes para construir el código y cambiar los nombres a unos más entendibles y prácticos


### b) Renombrar variables `rename()`

- Estructura del código:

```r
base_de_datos <- base_de_datos %>%

  dplyr::rename(
    nombre_nuevo = nombre_original,

    nombre_nuevo = nombre_original,

    nombre_nuevo = nombre_original

  )
```

In [ ]:
base_antropologia <- base_antropologia %>%
  dplyr::rename(
    edad = p02_edad_del_a_entrevistado,
    genero = p03_genero_del_a_entrevistado_a,
    anio_carrera = p04_ano_en_que_se_encuentra_de_la_carrera_1_2_3_4_5,
    comuna_actual = p05_comuna_actual_de_residencia,
    comuna_previa = p06_comuna_de_residencia_de_su_familia_nuclear_padres_hermanos_as_u_otros_as_cuidadores_o_en_la_que_vivio_la_mayor_parte_de_infancia_y_adolescencia,
    tipo_establecimiento = p07_ultimo_tipo_de_establecimiento_educativo_en_que_realizo_su_ensenanza_media,
    puntaje = p08_puntaje_final_obtenido_en_la_prueba_de_seleccion_universitaria_poderado_segun_carrera_elegida,
    situacion = p09_cual_de_estas_situaciones_describe_mejor_su_actividad_principal_durante_el_ultimo_mes,
    nivel_educativo_madre = p10_indique_el_maximo_nivel_educativo_obtenido_por_su_madre,
    empleo_madre = p11_actualmente_su_madre_trabaja,
    ocupacion_madre = p12_cual_es_la_ocupacion_u_oficio_actual_de_su_madre_describa_las_principales_tareas_y_funciones_en_el_puesto_de_trabajo_actual_de_su_madre,
    nivel_educativo_padre = p13_indique_el_maximo_nivel_educativo_obtenido_por_su_padre,
    empleo_padre = p14_actualmente_su_padre_trabaja,
    ocupacion_padre = p15_cual_es_la_ocupacion_u_oficio_actual_de_su_padre_describa_las_principales_tareas_y_funciones_en_el_puesto_de_trabajo_actual_de_su_padre,
    sostenedor = p17_quien_es_el_principal_sostenedor_a_de_su_hogar_actual_quien_aporta_mas_ingresos,
    clase_social = p18_en_la_sociedad_comunmente_existen_distintos_grupos_o_clases_sociales_las_personas_de_clase_social_alta_son_las_que_tienen_los_ingresos_mas_altos_el_mayor_nivel_de_educacion_y_los_trabajos_mas_valorados_las_personas_de_clase_social_baja_son_las_que_tienen_los_ingresos_mas_bajos_el_menor_nivel_de_educacion_y_los_trabajos_menos_valorados_entre_estas_clases_existen_otras_intermedias_segun_su_opinion_a_cual_de_los_siguientes_grupos_o_clases_sociales_pertenece_usted,
    acceso_computador_hogar = p19_podria_decirme_si_su_casa_tiene_computador_ya_sea_notebook_o_de_escritorio_actualmente,
    acceso_computador_personal = p20_podria_decirme_si_usted_tiene_computador_para_uso_personal_ya_sea_notebook_o_de_escritorio_actualmente,
    acceso_celular = p21_podria_decirme_si_usted_tiene_smartphone_personal_actualmente,
    frecuencia_música = p22_con_que_frecuencia_escucha_musica,
    preferencia_música_1 = p23_que_tipo_de_musica_es_la_que_mas_prefiere_escuchar_aun_cuando_escuche_mas_de_un_estilo_elija_el_que_mas_escuche,
    preferencia_música_otra_1 = p24_si_eligio_otra_cual,
    preferencia_música_2 = p25_cual_es_la_segunda_musica_que_mas_prefiere_escuchar,
    preferencia_música_otra_2 = p26_si_eligio_otra_cual,
    dispositivo_música = p27_con_que_dispositivo_suele_escuchar_mas_musica,
    app_música = p28_cual_es_principal_sitio_programa_o_aplicacion_para_bajar_o_escuchar_musica,
    app_música_otra = p29_si_respondio_otro_cual,
    red_social_tiempo_1 = p30_cual_es_la_red_social_pasa_mas_tiempo,
    red_social_tiempo_otra_1 = p31_si_respondio_otra_cual,
    red_social_tiempo_2 = p32_y_cual_es_la_segunda_red_social_pasa_mas_tiempo,
    red_social_tiempo_otra_2 = p33_si_respondio_otra_cual
  )

names(base_antropologia)

## 3. Transformación de datos: Recodificación de variables y sus **categorías de respuesta** `recode` y `case_when`

### a) Variables cualitativas

In [ ]:
sapply(base_antropologia, FUN = unique)

### Exploramos la variable de interés: ocupación de la madre

In [ ]:
table(base_antropologia$ocupacion_madre)

### Homogeneizar valores

In [ ]:
base_antropologia <- base_antropologia %>%
  mutate(
    ocupacion_madre = stringi::stri_trans_general(ocupacion_madre, "Latin-ASCII"), #elimina tildes y transforma caracteres del sistema latino a caracteres ASCII (ñ)
    ocupacion_madre = tolower(ocupacion_madre), #transforma todos los caracteres al minúsculas
    ocupacion_madre = gsub(" ", "_", ocupacion_madre) #Busca los espacios " " y reemplaza por guiones bajos "_"
  )

### Verificar categorías

In [ ]:
table(base_antropologia$ocupacion_madre)
unique(base_antropologia$ocupacion_madre)

### Recodificar categorías con `recode`

In [ ]:
base_antropologia <- base_antropologia %>%
  mutate(
    ocupacion_madre = recode(ocupacion_madre,
                             "reponedara_en_un_supermercado_" = "Servicios",
                             "docente_de_yoga" = "Servicios",
                             "reponedora" = "Servicios",
                             "paisajista,_esta_a_cargo_de_supervisar_y_dirigir_el_mantenimiento_de_las_areas_verdes_en_una_comuna." = "Servicios",
                             "tia_de_furgon,_transportar_ninos_de_basica\r\n" = "Servicios",
                             "duena_de_casa,_cuidados_del_hogar,,_repostera_" = "Servicios",
                             "cocinera_de_casino" = "Servicios",
                             "instructor_de_yoga_y_actividad_fisica_para_adultos_y_adultos_mayores,_tambien_trabaja_por_una_empresa_local_como_organizadora_de_casas._como_organizadora,_limpia_profundamente_y_ordena_casas,_bota_cosas_en_masa,_etc" = "Servicios",
                             "tens,_trabaja_en_el_pensionado_de_un_hospital_y_tiene_que_atender_personas_post_operatorios_o_casos_psiquiatricos_derivados_del_estado." = "Salud y cuidado",
                             "auxiliar_de_servicio" = "Salud y cuidado",
                             "abogada" = "Administrativo / Profesional",
                             "administracion_en_empresa_de_transporte_de_valores" = "Administrativo / Profesional",
                             "duena_de_casa" = "Trabajo doméstico / Inactiva",
                             "jubilada" = "Trabajo doméstico / Inactiva",
                             "-" = "Desconocido"
    )
  )

table(base_antropologia$ocupacion_madre)

### b) Variables numéricas: recodificación con `case_when`

In [ ]:
table(base_antropologia$puntaje)

In [ ]:
base_antropologia <- base_antropologia %>%
  mutate(puntaje = case_when(
    puntaje %in% c("650", "670", "680", "700+") ~ "Alto",
    puntaje %in% c("610", "630") ~ "Medio",
    puntaje %in% c("500", "590", "no me acuerdo, pero creo que eran como 590/600") ~ "Bajo",
    puntaje %in% c("No se", "No se aplica (ACT 30)") ~ "NA",
    TRUE ~ NA_character_
  ))

table(base_antropologia$puntaje)

## 🔶 Ejercicio: Recodificación de la variable `edad`

Completa el código para recodificar la variable `edad` utilizando `mutate` y `case_when`.

### 1. Recodificar

In [ ]:
base_antropologia <- _______________ %>%
  ______(
    edad_recodificada = _________(
      grepl("años", edad) ~ as.numeric(gsub(" años", "", edad)), # Elimina " años" y convierte a numérico
      TRUE ~ as.numeric(edad)
    )
  )

### 2. Verificar el resultado

In [ ]:
_____(base_antropologia$edad_recodificada)

## 4. Guardar base de datos limpia

In [ ]:
dir.create(path = "base limpia")
write.xlsx(x = base_antropologia, file = "base limpia/Encuesta_Antropología_Limpia.xlsx")

## 🔶 Ejercicio: guardar la base con otro nombre

Completa el código para guardar la base con el nombre **Datos**, en la carpeta llamada **Output**. ¿Cómo sería el código?

### 1. Crear carpeta llamada `Output`

In [ ]:
______ (path = "______")

### 2. Guardar base limpia con el nombre `Datos` en la carpeta `Output`

In [ ]:
______ (x = _____, file = "______/Datos.xlsx")

---

## Conclusiones

En esta ayudantía trabajamos un flujo básico de **data wrangling**:

- explorar una base de datos
- limpiar nombres de variables
- renombrar variables
- homogeneizar categorías
- recodificar variables cualitativas
- recodificar variables utilizando `case_when()`
- guardar una base de datos limpia en formato Excel